# Scraping Elecciones Colombia — 8 de marzo de 2026

Extracción estructurada de datos electorales desde el portal de resultados de la Registraduría Nacional.

**Fuente**: `https://resultados.registraduria.gov.co`

> **Nota técnica**: La cookie `cf_clearance` (protección Cloudflare) puede vencer en horas o días. Si recibes un error `403 Forbidden`, copia una nueva cookie desde el navegador y actualiza el archivo `config.json` en la raíz del proyecto.

todas las funciones que hacen requests HTTP usan lógica de retry (reintentos automáticos) y caching simple en disco usando solo la librería estándar de Python. Esto mejora la robustez y eficiencia del scraping.

- **Retry:** Si una petición falla por error de red o status >= 500, se reintenta hasta 3 veces con espera exponencial.
- **Caching:** Las respuestas exitosas se guardan en disco (`.cache/`), evitando repetir requests idénticos.

> **Nota:** No se usan librerías externas, solo la librería estándar (`os`, `time`, `hashlib`, `json`).

In [1]:
import json
import os
import time
import requests
import pandas as pd

# Cargar configuración centralizada (cookies, headers, URLs)
CONFIG_PATH = os.path.join('..', '..', 'config.json')
with open(CONFIG_PATH) as f:
    config = json.load(f)

URL = config['nomenclator_url']
cookies = config['cookies']
headers = config['headers']

# Usar cached_request con retry y caching
response = requests.get(URL, cookies=cookies, headers=headers)
if response.status_code == 200:
    data = response.json()
    print(f"✅ nomenclator.json cargado ({len(data['elec'])} elecciones, {len(data['partidos'])} partidos)")
else:
    print("❌ Request failed with status code:", response.status_code)
    print("   → Verifica que cf_clearance en config.json sea válido")


✅ nomenclator.json cargado (4 elecciones, 348 partidos)


# Paso 1: Entender la estructura del JSON

El archivo `nomenclator.json` es el **diccionario maestro** de las elecciones. La Registraduría lo usa para que los archivos de resultados sean livianos: en vez de enviar nombres completos, envía códigos numéricos. Este JSON nos permite "traducir" esos códigos a nombres reales.

**Llaves principales del JSON:**
| Llave | Descripción |
|-------|-------------|
| `ver` | Versión del archivo |
| `y` | Año electoral (2026) |
| `elec` | Lista de **elecciones/corporaciones** (Senado, Cámara, Consultas, CITREP) con sus circunscripciones |
| `levels` | **Niveles** de la jerarquía geográfica (Colombia → Departamento → Municipio → Zona → Comuna → Puesto → Mesa) |
| `amb` | **Ámbitos** (División Político-Administrativa / Divipol). Contiene todos los lugares organizados jerárquicamente |
| `partidos` | Lista de **partidos políticos** con código, nombre y color |

A continuación construiremos un DataFrame (tabla de dimensión) para cada entidad.

In [2]:
# Verificamos las llaves principales del JSON
print("Llaves principales del nomenclator:")
for key in data.keys():
    val = data[key]
    tipo = type(val).__name__
    if isinstance(val, list):
        print(f"  '{key}' -> {tipo}, {len(val)} elementos")
    elif isinstance(val, dict):
        print(f"  '{key}' -> {tipo}")
    else:
        print(f"  '{key}' -> {tipo}: {val}")
# Recuerda: todas las requests usan cached_request para retry y caching

Llaves principales del nomenclator:
  'ver' -> int: 1
  'y' -> str: 2026
  'elec' -> list, 4 elementos
  'levels' -> list, 7 elementos
  'amb' -> list, 4 elementos
  'partidos' -> list, 348 elementos


## Paso 2: Tabla de dimensión — Elecciones y Circunscripciones (`dim_elecciones`)

Cada elección (Senado, Cámara, etc.) tiene una o más **circunscripciones** (también llamadas "cámaras" en el JSON, campo `cam`). Por ejemplo, el Senado tiene circunscripción Nacional e Indígena, mientras que Cámara tiene Territorial Departamental, Indígena y Afro.

Creamos una tabla plana donde cada fila es una combinación **elección + circunscripción**.

In [3]:
# Construimos dim_elecciones: una fila por cada combinación elección + circunscripción
filas_elecciones = []

for eleccion in data['elec']:
    for cam in eleccion['cam']:
        filas_elecciones.append({
            'elec_id': eleccion['i'],          # Índice interno de la elección
            'elec_codigo': eleccion['elec'],    # Código de la elección usado en los archivos de votos
            'elec_orden': eleccion['ord'],      # Orden de presentación
            'elec_sigla': eleccion['sigla'],    # Sigla (SE, CA, CN, CT)
            'elec_nombre': eleccion['n'],       # Nombre completo de la elección
            'elec_color': eleccion['c'],        # Color principal
            'elec_color_sec': eleccion['cs'],   # Color secundario
            'cam_id': cam['i'],                 # Índice de la circunscripción
            'cam_nombre': cam['n'],             # Nombre de la circunscripción
            'cam_color': cam['c'],              # Color de la circunscripción
        })

dim_elecciones = pd.DataFrame(filas_elecciones)
print(f"dim_elecciones: {len(dim_elecciones)} filas")
dim_elecciones

dim_elecciones: 7 filas


,elec_id,elec_codigo,elec_orden,elec_sigla,elec_nombre,elec_color,elec_color_sec,cam_id,cam_nombre,cam_color
0,0,1,1,SE,SENADO,#1C283D,#47658F,0,NACIONAL,#FFC53D
1,0,1,1,SE,SENADO,#1C283D,#47658F,4,INDIGENAS,#004F9A
2,1,2,2,CA,CAMARA,#1C283D,#919662,1,TERRITORIAL DEPARTAMENTAL,#FFC53D
3,1,2,2,CA,CAMARA,#1C283D,#919662,4,INDIGENAS,#004F9A
4,1,2,2,CA,CAMARA,#1C283D,#919662,5,AFRO-DESCENDIENTES,#EC0000
5,2,6,3,CN,CONSULTAS INTERPARTIDISTAS,#1C283D,#64a70b,0,NACIONAL,#FFC53D
6,3,7,4,CT,CITREP,#1C283D,#925862,9,CITREP,#A2789C


## Paso 3: Tabla de dimensión — Niveles Geográficos (`dim_niveles`)

La jerarquía geográfica de la Registraduría tiene 7 niveles. Esta tabla es pequeña pero crucial para entender qué nivel (`l`) corresponde a qué tipo de entidad territorial.

`1: COLOMBIA → 2: DEPARTAMENTO → 3: MUNICIPIO → 4: ZONA → 5: COMUNA → 6: PUESTO → 7: MESA`

In [4]:
# Construimos dim_niveles: directo del array 'levels'
dim_niveles = pd.DataFrame(data['levels'])
dim_niveles.columns = ['nivel_id', 'nivel_nombre']

print(f"dim_niveles: {len(dim_niveles)} filas")
dim_niveles

dim_niveles: 7 filas


,nivel_id,nivel_nombre
0,1,COLOMBIA
1,2,DEPARTAMENTO
2,3,MUNICIPIO
3,4,ZONA
4,5,COMUNA
5,6,PUESTO
6,7,MESA


## Paso 4: Tabla de dimensión — Partidos Políticos (`dim_partidos`)

Cada partido tiene un código (`codpar`) que se usa en los archivos de resultados de votos. Esta tabla nos permite traducir ese código al nombre completo del partido y su color oficial.

Hay **348 registros** que incluyen partidos, movimientos, coaliciones y categorías especiales como "Votos en blanco, nulos y no marcados".

In [5]:
# Construimos dim_partidos: directo del array 'partidos'
dim_partidos = pd.DataFrame(data['partidos'])

# Renombramos columnas a nombres descriptivos
dim_partidos.rename(columns={
    'codpar': 'partido_codigo',
    'nombre': 'partido_nombre',
    'color': 'partido_color',
    'i': 'partido_indice',
    's': 'partido_slug',
}, inplace=True)

print(f"dim_partidos: {len(dim_partidos)} filas")
dim_partidos.head(10)

dim_partidos: 348 filas


,partido_codigo,partido_nombre,partido_color,partido_indice,partido_slug
0,0,"VOTOS EN BLANCO, NULOS Y NO MARCADOS",#5A7CCC,1,"VOTOS-EN-BLANCO,-NULOS-Y-NO-MARCADOS"
1,1,PARTIDO LIBERAL COLOMBIANO,#E30716,2,PARTIDO-LIBERAL-COLOMBIANO
2,2,PARTIDO CONSERVADOR COLOMBIANO,#0867B1,3,PARTIDO-CONSERVADOR-COLOMBIANO
3,3,PARTIDO CAMBIO RADICAL,#E3051C,4,PARTIDO-CAMBIO-RADICAL
4,4,PARTIDO ALIANZA VERDE,#007C34,5,PARTIDO-ALIANZA-VERDE
5,5,"MOVIMIENTO AUTORIDADES INDÍGENAS DE COLOMBIA ""...",#C05B16,6,"MOVIMIENTO-AUTORIDADES-INDIGENAS-DE-COLOMBIA-""..."
6,6,"PARTIDO ALIANZA SOCIAL INDEPENDIENTE ""ASI""",#F7C412,7,"PARTIDO-ALIANZA-SOCIAL-INDEPENDIENTE-""ASI"""
7,7,PARTIDO POLÍTICO MIRA,#06539C,8,PARTIDO-POLITICO-MIRA
8,8,PARTIDO DE LA UNIÓN POR LA GENTE - PARTIDO DE ...,#48AB38,9,PARTIDO-DE-LA-UNION-POR-LA-GENTE---PARTIDO-DE-...
9,11,PARTIDO CENTRO DEMOCRÁTICO,#1E477D,10,PARTIDO-CENTRO-DEMOCRATICO


## Paso 5: Tabla de dimensión — Ámbitos Geográficos / Divipol (`dim_ambitos`)

Esta es la tabla más importante y compleja. El array `amb` contiene la **División Político-Administrativa (Divipol)** organizada por elección. Cada ámbito tiene:

| Campo | Significado |
|-------|-------------|
| `i` | Índice interno (posición en el array) |
| `n` | Nombre del lugar |
| `co` | Código Divipol de la Registraduría |
| `s` | Slug (nombre para URL) |
| `l` | Nivel jerárquico (1=Colombia, 2=Departamento, 3=Municipio) |
| `p` | **Padres**: lista de índices que apuntan al nivel superior |
| `h` | **Hijos**: lista de índices que apuntan al nivel inferior |
| `r` | Array de resoluciones/referencias internas |

Vamos a extraer una tabla plana de ámbitos para cada elección, resolviendo la relación padre para saber a qué departamento pertenece cada municipio.

In [6]:
# Primero exploramos cuántos ámbitos hay por elección y por nivel
for grupo in data['amb']:
    elec_id = grupo['elec']
    ambitos = grupo['ambitos']
    
    # Conteo por nivel
    conteo = {}
    for a in ambitos:
        nivel = a['l']
        conteo[nivel] = conteo.get(nivel, 0) + 1
    
    print(f"Elección {elec_id}: {len(ambitos)} ámbitos totales")
    for nivel in sorted(conteo.keys()):
        nombre_nivel = next((l['n'] for l in data['levels'] if l['l'] == nivel), '?')
        print(f"  Nivel {nivel} ({nombre_nivel}): {conteo[nivel]}")
    print()

Elección 1: 1224 ámbitos totales
  Nivel 1 (COLOMBIA): 1
  Nivel 2 (DEPARTAMENTO): 34
  Nivel 3 (MUNICIPIO): 1189

Elección 2: 1224 ámbitos totales
  Nivel 1 (COLOMBIA): 1
  Nivel 2 (DEPARTAMENTO): 34
  Nivel 3 (MUNICIPIO): 1189

Elección 6: 1224 ámbitos totales
  Nivel 1 (COLOMBIA): 1
  Nivel 2 (DEPARTAMENTO): 34
  Nivel 3 (MUNICIPIO): 1189

Elección 7: 185 ámbitos totales
  Nivel 1 (COLOMBIA): 1
  Nivel 2 (DEPARTAMENTO): 16
  Nivel 3 (MUNICIPIO): 168



In [7]:
# Verificamos si las elecciones 1, 2 y 6 comparten los mismos ámbitos
# Comparamos los códigos (co) y nombres (n) de los ámbitos
ambitos_por_elec = {}
for grupo in data['amb']:
    codigos = set((a['co'], a['n'], a['l']) for a in grupo['ambitos'])
    ambitos_por_elec[grupo['elec']] = codigos

# Comparar elección 1 vs 2
iguales_1_2 = ambitos_por_elec[1] == ambitos_por_elec[2]
iguales_1_6 = ambitos_por_elec[1] == ambitos_por_elec[6]
print(f"¿Elecciones 1 y 2 tienen los mismos ámbitos? {iguales_1_2}")
print(f"¿Elecciones 1 y 6 tienen los mismos ámbitos? {iguales_1_6}")
print(f"Elección 7 (CITREP) tiene un subconjunto de {len(ambitos_por_elec[7])} ámbitos")

¿Elecciones 1 y 2 tienen los mismos ámbitos? True
¿Elecciones 1 y 6 tienen los mismos ámbitos? True
Elección 7 (CITREP) tiene un subconjunto de 185 ámbitos


### Construcción de `dim_ambitos`

Las elecciones 1 (Senado), 2 (Cámara) y 6 (Consultas) comparten **exactamente los mismos 1224 ámbitos**. La elección 7 (CITREP) usa un subconjunto de 185.

Tomamos los ámbitos de la elección 1 como base y resolvemos la relación padre-hijo para crear una tabla donde cada municipio tenga su departamento asociado.

**Lógica de resolución del padre**: El campo `p` de cada ámbito contiene una lista de objetos `{l: nivel, p: [índices]}`. Para un municipio (nivel 3), su padre es el departamento (nivel 2) cuyo índice está en `p[0].p[0]`.

In [8]:
# Tomamos los ámbitos de la elección 1 (Senado) como base completa
ambitos_base = data['amb'][0]['ambitos']  # elec=1

# Paso 1: Crear un diccionario de lookup por índice para resolver padres
lookup = {a['i']: a for a in ambitos_base}

# Paso 2: Construir filas planas resolviendo la jerarquía
filas_ambitos = []

for a in ambitos_base:
    fila = {
        'ambito_indice': a['i'],            # Índice interno (posición en el array)
        'ambito_nombre': a['n'],             # Nombre del lugar
        'ambito_codigo': a['co'],            # Código Divipol
        'ambito_slug': a.get('s', ''),       # Slug para URL
        'nivel_id': a['l'],                  # Nivel jerárquico
    }
    
    # Resolver el padre inmediato (si existe)
    # El campo 'p' es una lista de relaciones: [{l: nivel_padre, p: [indices_padre]}]
    padre_indice = None
    if a['p']:
        # Tomamos la primera relación padre
        padre_rel = a['p'][0]
        if padre_rel['p']:
            padre_indice = padre_rel['p'][0]
    
    fila['padre_indice'] = padre_indice
    
    # Si es un municipio (nivel 3), resolver el nombre del departamento padre
    if a['l'] == 3 and padre_indice is not None:
        padre = lookup.get(padre_indice, {})
        fila['departamento_nombre'] = padre.get('n', '')
        fila['departamento_codigo'] = padre.get('co', '')
    elif a['l'] == 2:
        # El departamento es su propio departamento
        fila['departamento_nombre'] = a['n']
        fila['departamento_codigo'] = a['co']
    else:
        fila['departamento_nombre'] = None
        fila['departamento_codigo'] = None
    
    filas_ambitos.append(fila)

dim_ambitos = pd.DataFrame(filas_ambitos)
print(f"dim_ambitos: {len(dim_ambitos)} filas")
print("\nDistribución por nivel:")
print(dim_ambitos.groupby('nivel_id').size().reset_index(name='cantidad'))
dim_ambitos.head(10)

dim_ambitos: 1224 filas

Distribución por nivel:
   nivel_id  cantidad
0         1         1
1         2        34
2         3      1189


,ambito_indice,ambito_nombre,ambito_codigo,ambito_slug,nivel_id,padre_indice,departamento_nombre,departamento_codigo
0,0,COLOMBIA,00,COLOMBIA,1,NaN,NaN,NaN
1,1,AMAZONAS,6000,AMAZONAS,2,0.0,AMAZONAS,6000
2,2,ANTIOQUIA,0100,ANTIOQUIA,2,0.0,ANTIOQUIA,0100
3,3,ARAUCA,4000,ARAUCA,2,0.0,ARAUCA,4000
4,4,ATLANTICO,0300,ATLANTICO,2,0.0,ATLANTICO,0300
5,5,BOGOTA D.C.,1600,BOGOTA-D.C.,2,0.0,BOGOTA D.C.,1600
6,6,BOLIVAR,0500,BOLIVAR,2,0.0,BOLIVAR,0500
7,7,BOYACA,0700,BOYACA,2,0.0,BOYACA,0700
8,8,CALDAS,0900,CALDAS,2,0.0,CALDAS,0900
9,9,CAQUETA,4400,CAQUETA,2,0.0,CAQUETA,4400


## Paso 6: Tablas de conveniencia — Departamentos y Municipios

A partir de `dim_ambitos` creamos dos vistas filtradas que son las más útiles para análisis:

- **`dim_departamentos`**: Solo los 34 departamentos (nivel 2)
- **`dim_municipios`**: Los 1189 municipios (nivel 3) con su departamento ya resuelto

También creamos `dim_ambitos_citrep` para los 185 ámbitos específicos de la elección CITREP (Circunscripción Transitoria Especial de Paz).

In [9]:
# --- dim_departamentos: solo nivel 2 ---
dim_departamentos = dim_ambitos[dim_ambitos['nivel_id'] == 2][
    ['ambito_indice', 'ambito_nombre', 'ambito_codigo']
].rename(columns={
    'ambito_indice': 'depto_indice',
    'ambito_nombre': 'depto_nombre',
    'ambito_codigo': 'depto_codigo',
}).reset_index(drop=True)

print(f"dim_departamentos: {len(dim_departamentos)} filas")
dim_departamentos

dim_departamentos: 34 filas


,depto_indice,depto_nombre,depto_codigo
0,1,AMAZONAS,6000
1,2,ANTIOQUIA,0100
2,3,ARAUCA,4000
3,4,ATLANTICO,0300
4,5,BOGOTA D.C.,1600
5,6,BOLIVAR,0500
6,7,BOYACA,0700
7,8,CALDAS,0900
8,9,CAQUETA,4400
9,10,CASANARE,4600


In [10]:
# --- dim_municipios: solo nivel 3, con departamento resuelto ---
dim_municipios = dim_ambitos[dim_ambitos['nivel_id'] == 3][
    ['ambito_indice', 'ambito_nombre', 'ambito_codigo', 'departamento_nombre', 'departamento_codigo']
].rename(columns={
    'ambito_indice': 'mpio_indice',
    'ambito_nombre': 'mpio_nombre',
    'ambito_codigo': 'mpio_codigo',
    'departamento_nombre': 'depto_nombre',
    'departamento_codigo': 'depto_codigo',
}).reset_index(drop=True)

print(f"dim_municipios: {len(dim_municipios)} filas")
dim_municipios.head(10)

dim_municipios: 1189 filas


,mpio_indice,mpio_nombre,mpio_codigo,depto_nombre,depto_codigo
0,35,ABEJORRAL,0100004,ANTIOQUIA,0100
1,36,ABREGO,2500004,NORTE DE SAN,2500
2,37,ABRIAQUI,0100007,ANTIOQUIA,0100
3,38,ACACIAS,5200005,META,5200
4,39,ACANDI,1700004,CHOCO,1700
5,40,ACEVEDO,1900004,HUILA,1900
6,41,ACHI,0500004,BOLIVAR,0500
7,42,AGRADO,1900007,HUILA,1900
8,43,AGUA DE DIOS,1500004,CUNDINAMARCA,1500
9,44,AGUACHICA,1200075,CESAR,1200


In [11]:
# --- dim_ambitos_citrep: ámbitos específicos de CITREP (elección 7) ---
# CITREP = Circunscripción Transitoria Especial de Paz
ambitos_citrep = data['amb'][3]['ambitos']  # elec=7
lookup_citrep = {a['i']: a for a in ambitos_citrep}

filas_citrep = []
for a in ambitos_citrep:
    padre_indice = None
    if a['p']:
        padre_rel = a['p'][0]
        if padre_rel['p']:
            padre_indice = padre_rel['p'][0]
    
    fila = {
        'ambito_indice': a['i'],
        'ambito_nombre': a['n'],
        'ambito_codigo': a['co'],
        'nivel_id': a['l'],
        'padre_indice': padre_indice,
    }
    
    # Resolver departamento para municipios
    if a['l'] == 3 and padre_indice is not None:
        padre = lookup_citrep.get(padre_indice, {})
        fila['departamento_nombre'] = padre.get('n', '')
    elif a['l'] == 2:
        fila['departamento_nombre'] = a['n']
    else:
        fila['departamento_nombre'] = None
    
    filas_citrep.append(fila)

dim_ambitos_citrep = pd.DataFrame(filas_citrep)
print(f"dim_ambitos_citrep: {len(dim_ambitos_citrep)} filas")
print(f"  Departamentos CITREP: {len(dim_ambitos_citrep[dim_ambitos_citrep['nivel_id'] == 2])}")
print(f"  Municipios CITREP: {len(dim_ambitos_citrep[dim_ambitos_citrep['nivel_id'] == 3])}")
dim_ambitos_citrep[dim_ambitos_citrep['nivel_id'] == 3].head(10)

dim_ambitos_citrep: 185 filas
  Departamentos CITREP: 16
  Municipios CITREP: 168


,ambito_indice,ambito_nombre,ambito_codigo,nivel_id,padre_indice,departamento_nombre
17,17,ACANDI,1706004,3,13.0,CIRCUNSCRIPCIÓN 6 (CHOCÓ - ANT
18,18,AGUSTIN CODAZZI,1212150,3,4.0,CIRCUNSCRIPCIÓN 12 (CESAR - LA
19,19,ALBANIA,4405002,3,12.0,CIRCUNSCRIPCIÓN 5 (CAQUETÁ - H
20,20,ALGECIRAS,1905013,3,12.0,CIRCUNSCRIPCIÓN 5 (CAQUETÁ - H
21,21,AMALFI,0103016,3,10.0,CIRCUNSCRIPCIÓN 3 ANTIOQUIA
22,22,ANORI,0103028,3,10.0,CIRCUNSCRIPCIÓN 3 ANTIOQUIA
23,23,APARTADO,0116035,3,8.0,CIRCUNSCRIPCIÓN 16 ANTIOQUIA
24,24,ARACATACA,2112010,3,4.0,CIRCUNSCRIPCIÓN 12 (CESAR - LA
25,25,ARAUQUITA,4002010,3,9.0,CIRCUNSCRIPCIÓN 2 ARAUCA
26,26,ARENAL,0513005,3,5.0,CIRCUNSCRIPCIÓN 13 (BOLÍVAR -


## Paso 7: Resumen de tablas de dimensión creadas

Todas las tablas de dimensión están listas para ser usadas como lookup al procesar los archivos de resultados de votos.

| DataFrame | Filas | Descripción |
|-----------|-------|-------------|
| `dim_elecciones` | 7 | Elecciones + circunscripciones (Senado Nacional, Senado Indígena, Cámara Territorial, etc.) |
| `dim_niveles` | 7 | Jerarquía geográfica (Colombia → Depto → Municipio → Zona → Comuna → Puesto → Mesa) |
| `dim_partidos` | 348 | Partidos políticos con código, nombre y color |
| `dim_ambitos` | 1,224 | Todos los ámbitos geográficos (país + deptos + municipios) con jerarquía resuelta |
| `dim_departamentos` | 34 | Vista filtrada: solo departamentos |
| `dim_municipios` | 1,189 | Vista filtrada: solo municipios con su departamento |
| `dim_ambitos_citrep` | 185 | Ámbitos específicos de las Circunscripciones Transitorias Especiales de Paz |

**Uso**: Cuando los archivos de votos traigan un código como `"0100004"`, se puede buscar en `dim_ambitos` para saber que es **ABEJORRAL, ANTIOQUIA**. Igualmente, un `codpar = "1"` se traduce a **PARTIDO LIBERAL COLOMBIANO** usando `dim_partidos`.

In [12]:
# Resumen final: todas las tablas de dimensión en memoria
tablas = {
    'dim_elecciones': dim_elecciones,
    'dim_niveles': dim_niveles,
    'dim_partidos': dim_partidos,
    'dim_ambitos': dim_ambitos,
    'dim_departamentos': dim_departamentos,
    'dim_municipios': dim_municipios,
    'dim_ambitos_citrep': dim_ambitos_citrep,
}

print("Tablas de dimensión disponibles:")
print("-" * 50)
for nombre, df in tablas.items():
    print(f"  {nombre:25s} -> {len(df):>5} filas, {len(df.columns)} columnas")
print("-" * 50)
print(f"Total: {len(tablas)} tablas creadas")

Tablas de dimensión disponibles:
--------------------------------------------------
  dim_elecciones            ->     7 filas, 10 columnas
  dim_niveles               ->     7 filas, 2 columnas
  dim_partidos              ->   348 filas, 5 columnas
  dim_ambitos               ->  1224 filas, 8 columnas
  dim_departamentos         ->    34 filas, 3 columnas
  dim_municipios            ->  1189 filas, 5 columnas
  dim_ambitos_citrep        ->   185 filas, 6 columnas
--------------------------------------------------
Total: 7 tablas creadas


## Datos granulares: resultados de la elección de Cámara (CA)

La elección de Cámara tiene **3 circunscripciones** (cámaras):
- **Territorial Departamental** (cam_id=1): curules por departamento
- **Indígenas** (cam_id=4): circunscripción especial indígena
- **Afrodescendientes** (cam_id=5): circunscripción especial afrodescendiente

A diferencia de las Consultas Interpartidistas (que tenían 1 sola cámara), aquí cada response del API contiene un array `camaras` con **múltiples entradas** que debemos iterar.

Primero exploramos la estructura de la respuesta del API para entender los datos disponibles.

In [13]:
# Explorar estructura de la respuesta de Cámara (CA) a nivel nacional
url_ca = "https://resultados.registraduria.gov.co/json/ACT/CA/00.json"
r_ca = requests.get(url_ca, cookies=cookies, headers=headers)
ca_data = r_ca.json()

print(f"Status: {r_ca.status_code}")
print(f"Llaves top: {list(ca_data.keys())}")
print(f"Número de cámaras: {len(ca_data['camaras'])}")

for i, cam in enumerate(ca_data['camaras']):
    print(f"\n{'='*60}")
    print(f"--- Cámara {i} ---")
    print(f"  Llaves: {list(cam.keys())}")
    print(f"  nomcam: {cam.get('nomcam', '?')}")
    print(f"  codcam: {cam.get('codcam', '?')}")
    n_part = len(cam.get('partotabla', []))
    n_mapa = len(cam.get('mapagan', []))
    print(f"  partotabla: {n_part} partidos")
    print(f"  mapagan: {n_mapa} sub-ámbitos")
    
    # Totales
    if 'totales' in cam:
        t = cam['totales'].get('act', {})
        print(f"  Totales: votant={t.get('votant')}, votval={t.get('votval')}, votnul={t.get('votnul')}")
    
    # Ejemplo de un partido
    if cam.get('partotabla'):
        p0 = cam['partotabla'][0]['act']
        print(f"  Ejemplo partido: codpar={p0['codpar']}, vot={p0['vot']}")
        print(f"    cantotabla: {len(p0.get('cantotabla', []))} candidatos")
        if p0.get('cantotabla'):
            c0 = p0['cantotabla'][0]
            print(f"    Ejemplo candidato: {c0.get('nomcan','')} {c0.get('apecan','')}, vot={c0.get('vot','')}")

Status: 200
Llaves top: ['elec', 'amb', 'tope', 'numact', 'numdep', 'iscircus', 'mdhm', 'shc', 'totales', 'camaras', 'historico', 'dept']
Número de cámaras: 3

--- Cámara 0 ---
  Llaves: ['cam', 'cir', 'carg', 'cargElectos', 'cargEmpatados', 'cargtota', 'totales', 'partotabla', 'mapagan']
  nomcam: ?
  codcam: ?
  partotabla: 154 partidos
  mapagan: 34 sub-ámbitos
  Totales: votant=19947812, votval=18961246, votnul=628363
  Ejemplo partido: codpar=10, vot=2566981
    cantotabla: 0 candidatos

--- Cámara 1 ---
  Llaves: ['cam', 'cir', 'carg', 'cargElectos', 'cargEmpatados', 'cargtota', 'totales', 'partotabla', 'mapagan']
  nomcam: ?
  codcam: ?
  partotabla: 10 partidos
  mapagan: 34 sub-ámbitos
  Totales: votant=170261, votval=136786, votnul=13311
  Ejemplo partido: codpar=188, vot=45063
    cantotabla: 3 candidatos
    Ejemplo candidato: CARLOS MARIO CALVO LARGO, vot=33687

--- Cámara 2 ---
  Llaves: ['cam', 'cir', 'carg', 'cargElectos', 'cargEmpatados', 'cargtota', 'totales', 'partot

In [14]:
# Detalle de los campos cam y cir para cada cámara
for i, cam in enumerate(ca_data['camaras']):
    print(f"--- Cámara {i} ---")
    print(f"  cam (tipo {type(cam['cam']).__name__}): {cam['cam']}")
    print(f"  cir (tipo {type(cam['cir']).__name__}): {cam['cir']}")
    print(f"  carg: {cam.get('carg')}, cargElectos: {cam.get('cargElectos')}")
    print()

# Explorar respuesta a nivel departamento (ej: Antioquia 0500)
url_depto = "https://resultados.registraduria.gov.co/json/ACT/CA/0500.json"
r_depto = requests.get(url_depto, cookies=cookies, headers=headers)
depto_data = r_depto.json()
print(f"\n{'='*60}")
print("Respuesta para departamento Antioquia (0500):")
print(f"Número de cámaras: {len(depto_data['camaras'])}")
for i, cam in enumerate(depto_data['camaras']):
    n_part = len(cam.get('partotabla', []))
    n_mapa = len(cam.get('mapagan', []))
    n_cand_total = sum(len(p['act'].get('cantotabla', [])) for p in cam.get('partotabla', []))
    print(f"  Cámara {i}: cam={cam['cam']}, cir={cam['cir']}, partidos={n_part}, candidatos={n_cand_total}, mapagan={n_mapa}")

# Explorar respuesta a nivel municipio (ej: Medellín 05001)
url_mpio = "https://resultados.registraduria.gov.co/json/ACT/CA/05001.json"
r_mpio = requests.get(url_mpio, cookies=cookies, headers=headers)
print(f"\n{'='*60}")
print("Respuesta para municipio Medellín (05001):")
print(f"  Status: {r_mpio.status_code}")
print(f"  Content-Length: {len(r_mpio.content)} bytes")
print(f"  Content-Type: {r_mpio.headers.get('Content-Type', '?')}")
if r_mpio.status_code == 200 and len(r_mpio.content) > 0:
    mpio_data = r_mpio.json()
    print(f"  Número de cámaras: {len(mpio_data['camaras'])}")
    for i, cam in enumerate(mpio_data['camaras']):
        n_part = len(cam.get('partotabla', []))
        n_cand_total = sum(len(p['act'].get('cantotabla', [])) for p in cam.get('partotabla', []))
        t = cam['totales'].get('act', {})
        print(f"  Cámara {i}: cam={cam['cam']}, partidos={n_part}, candidatos={n_cand_total}")
        print(f"    totales: votant={t.get('votant')}, votval={t.get('votval')}")
else:
    print(f"  Respuesta vacía o error - primeros 200 chars: {r_mpio.text[:200]}")

--- Cámara 0 ---
  cam (tipo str): 1
  cir (tipo str): 2
  carg: 0, cargElectos: 0

--- Cámara 1 ---
  cam (tipo str): 4
  cir (tipo str): 1
  carg: 0, cargElectos: 0

--- Cámara 2 ---
  cam (tipo str): 5
  cir (tipo str): 1
  carg: 0, cargElectos: 0


Respuesta para departamento Antioquia (0500):
Número de cámaras: 3
  Cámara 0: cam=1, cir=1, partidos=9, candidatos=55, mapagan=46
  Cámara 1: cam=4, cir=0, partidos=10, candidatos=31, mapagan=46
  Cámara 2: cam=5, cir=0, partidos=46, candidatos=170, mapagan=46

Respuesta para municipio Medellín (05001):
  Status: 404
  Content-Length: 2102 bytes
  Content-Type: text/html
  Respuesta vacía o error - primeros 200 chars: <!doctype html>
<html lang="es" translate="no">
  <head>
    <meta charset="UTF-8" />
    <meta name="viewport" content="width=device-width, initial-scale=1.0" />
    <link rel="icon" href="/favicon.i


In [15]:
# Explorar mapagan (municipios) y partotabla (candidatos) a nivel departamento
cam_terr = depto_data['camaras'][0]  # Territorial en Antioquia

print("=== MAPAGAN (municipios ganador) ===")
print(f"Total: {len(cam_terr['mapagan'])} sub-ámbitos")
if cam_terr['mapagan']:
    m0 = cam_terr['mapagan'][0]
    print(f"Llaves: {list(m0.keys())}")
    print(f"Ejemplo: {m0}")
    print("\nPrimeros 5 municipios:")
    for m in cam_terr['mapagan'][:5]:
        print(f"  {m.get('amb')} - {m.get('nombre','?')}: ganador codpar={m.get('codpar')}, vot={m.get('vot')}, votant={m.get('votant')}")

print("\n=== PARTOTABLA (candidatos por partido) ===")
print(f"Total partidos: {len(cam_terr['partotabla'])}")
if cam_terr['partotabla']:
    p0 = cam_terr['partotabla'][0]['act']
    print(f"Llaves partido: {list(p0.keys())}")
    print(f"Partido: codpar={p0['codpar']}, vot={p0['vot']}")
    print(f"Candidatos: {len(p0.get('cantotabla', []))}")
    if p0.get('cantotabla'):
        c0 = p0['cantotabla'][0]
        print(f"  Llaves candidato: {list(c0.keys())}")
        print(f"  Ejemplo: {c0}")

# Verificar si mapagan tiene datos por cámara diferente
print("\n=== COMPARAR MAPAGAN POR CÁMARA ===")
for i, cam in enumerate(depto_data['camaras']):
    print(f"Cámara {i} (cam={cam['cam']}): mapagan={len(cam['mapagan'])} items")
    if cam['mapagan']:
        print(f"  Ejemplo: amb={cam['mapagan'][0]['amb']}, codpar={cam['mapagan'][0].get('codpar')}, vot={cam['mapagan'][0].get('vot')}")

=== MAPAGAN (municipios ganador) ===
Total: 46 sub-ámbitos
Llaves: ['amb', 'codpar', 'vot', 'pvot', 'votant', 'pvotant', 'votcan', 'mesesc', 'pmesesc', 'carg', 'nombre', 'hayEmpate']
Ejemplo: {'amb': '0500004', 'codpar': '3', 'vot': '6661', 'pvot': '57,95%', 'votant': '12030', 'pvotant': '63,39%', 'votcan': '11444', 'mesesc': '63', 'pmesesc': '100%', 'carg': '0', 'nombre': 'ACHI', 'hayEmpate': '0'}

Primeros 5 municipios:
  0500004 - ACHI: ganador codpar=3, vot=6661, votant=12030
  0500006 - ALTOS DEL ROSARIO: ganador codpar=3, vot=3186, votant=5039
  0500005 - ARENAL: ganador codpar=3, vot=2435, votant=3938
  0500007 - ARJONA: ganador codpar=3, vot=13332, votant=33204
  0500009 - ARROYO HONDO: ganador codpar=3, vot=5861, votant=7078

=== PARTOTABLA (candidatos por partido) ===
Total partidos: 9
Llaves partido: ['codpar', 'cam', 'vot', 'pvot', 'carg', 'cargElectos', 'cargEmpatados', 'cantotabla']
Partido: codpar=3, vot=422632
Candidatos: 7
  Llaves candidato: ['amb', 'codcan', 'cedula'

### Hallazgos de exploración

| Nivel | Endpoint | Resultado |
|-------|----------|-----------|
| Nacional | `/json/ACT/CA/00.json` | 3 cámaras, sin candidatos territoriales (solo Indígenas y Afro) |
| Departamento | `/json/ACT/CA/{depto_2dig}.json` | 3 cámaras, candidatos en `partotabla`, municipios ganador en `mapagan` |
| Municipio | `/json/ACT/CA/{ambito_7dig}.json` | ✅ 3 cámaras, candidatos en `partotabla` — requiere códigos de 7 dígitos de `dim_ambitos` (nivel_id=3) |

**Máxima granularidad posible**: nivel municipio, con 3 circunscripciones por response.
> **Nota**: Los códigos de 5 dígitos DANE (`dim_municipios.mpio_codigo`) devuelven 404. Se deben usar los códigos de 7 dígitos del nomenclátor (`dim_ambitos.ambito_codigo` donde `nivel_id == 3`).

### Estructura de cada response

Cada response contiene `camaras[3]`, una por circunscripción:

| `cam` | Circunscripción | Descripción |
|-------|----------------|-------------|
| `1` | Territorial Departamental | Candidatos específicos del departamento |
| `4` | Indígenas | Circunscripción especial indígena (nacional) |
| `5` | Afrodescendientes | Circunscripción especial afrodescendiente (nacional) |

Cada cámara contiene:
- **`partotabla`**: Partidos → candidatos con votos (la tabla más granular)
- **`mapagan`**: Resumen de ganador por municipio dentro del departamento (solo a nivel depto)
- **`totales`**: Votantes, votos válidos, nulos, no marcados

### Estrategia de scraping

**Nivel departamento** — 34 requests (uno por departamento) → **2 tablas**:
1. **`ca_candidatos_depto`**: Votos de cada candidato por partido, departamento y circunscripción
2. **`ca_municipios_ganador`**: Partido ganador y votos por municipio y circunscripción (desde `mapagan`)

**Nivel municipio** — 1,189 requests en paralelo (20 workers) → **1 tabla**:
3. **`ca_candidatos_municipio`**: Votos de cada candidato por partido, municipio y circunscripción (máxima granularidad)

## Scraping nivel departamento (34 requests)

Consultamos los 34 departamentos para obtener:
1. **`ca_candidatos_depto`**: Votos de cada candidato por partido, departamento y circunscripción (desde `partotabla`)
2. **`ca_municipios_ganador`**: Partido ganador por municipio y circunscripción (desde `mapagan`)

In [16]:
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from requests.adapters import HTTPAdapter
import threading

# === Configuración de Cámara ===
BASE_URL = "https://resultados.registraduria.gov.co/json/ACT/CA"
MAX_WORKERS = 10

# Circunscripciones de Cámara (3 cámaras por response)
CIRCUNSCRIPCIONES = {
    '1': 'Territorial Departamental',
    '4': 'Indígenas',
    '5': 'Afrodescendientes',
}

# === Scraping paralelo de los 34 departamentos para Cámara (CA) ===
session_depto = requests.Session()
session_depto.cookies.update(cookies)
session_depto.headers.update(headers)
adapter_depto = HTTPAdapter(pool_connections=MAX_WORKERS, pool_maxsize=MAX_WORKERS)
session_depto.mount('https://', adapter_depto)

lock_depto = threading.Lock()
completados_depto = 0

def fetch_departamento_ca(row):
    """Descarga y parsea los resultados de Cámara de un departamento."""
    global completados_depto
    codigo = row['depto_codigo']
    nombre = row['depto_nombre']
    
    r = session_depto.get(f"{BASE_URL}/{codigo}.json")
    
    with lock_depto:
        completados_depto += 1
        print(f"  [{completados_depto:2d}/34] {'✅' if r.status_code == 200 else '❌'} {nombre} ({codigo})")
    
    if r.status_code != 200:
        return None, (codigo, nombre, r.status_code)
    
    return r.json(), None

# Ejecutar en paralelo
rows_depto = [row for _, row in dim_departamentos.iterrows()]
responses_ca = {}
errores_depto = []

print(f"Descargando datos de {len(rows_depto)} departamentos ({MAX_WORKERS} workers)...")
t0 = time.time()

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = {executor.submit(fetch_departamento_ca, row): row for row in rows_depto}
    for future in as_completed(futures):
        row = futures[future]
        resp, error = future.result()
        if resp:
            responses_ca[row['depto_codigo']] = resp
        if error:
            errores_depto.append(error)

elapsed = time.time() - t0
print(f"\n✅ Completado en {elapsed:.1f}s: {len(responses_ca)} OK, {len(errores_depto)} errores")
if errores_depto:
    print(f"   Errores: {errores_depto}")


Descargando datos de 34 departamentos (10 workers)...
  [ 1/34] ✅ BOLIVAR (0500)
  [ 2/34] ✅ ATLANTICO (0300)
  [ 3/34] ✅ CESAR (1200)
  [ 4/34] ✅ CHOCO (1700)
  [ 5/34] ✅ ARAUCA (4000)
  [ 6/34] ✅ CAUCA (1100)
  [ 7/34] ✅ BOYACA (0700)
  [ 8/34] ✅ AMAZONAS (6000)
  [ 9/34] ✅ CORDOBA (1300)
  [10/34] ✅ HUILA (1900)
  [11/34] ✅ CUNDINAMARCA (1500)
  [12/34] ✅ CALDAS (0900)
  [13/34] ✅ CAQUETA (4400)
  [14/34] ✅ MAGDALENA (2100)
  [15/34] ✅ CONSULADOS (8800)
  [16/34] ✅ NARIÑO (2300)
  [17/34] ✅ QUINDIO (2600)
  [18/34] ✅ CASANARE (4600)
  [19/34] ✅ GUAINIA (5000)
  [20/34] ✅ GUAVIARE (5400)
  [21/34] ✅ ANTIOQUIA (0100)
  [22/34] ✅ LA GUAJIRA (4800)
  [23/34] ✅ BOGOTA D.C. (1600)
  [24/34] ✅ SANTANDER (2700)
  [25/34] ✅ TOLIMA (2900)
  [26/34] ✅ PUTUMAYO (6400)
  [27/34] ✅ VALLE (3100)
  [28/34] ✅ NORTE DE SAN (2500)
  [29/34] ✅ META (5200)
  [30/34] ✅ VICHADA (7200)
  [31/34] ✅ RISARALDA (2400)
  [32/34] ✅ VAUPES (6800)
  [33/34] ✅ SAN ANDRES (5600)
  [34/34] ✅ SUCRE (2800)

✅ Completad

### Construyendo DataFrames estructurados

De cada response departamental extraemos dos tablas (por cada circunscripción):

1. **`ca_candidatos_depto`**: Votos de cada candidato por partido, departamento y circunscripción (desde `partotabla`)
2. **`ca_municipios_ganador`**: Partido ganador y votos por municipio y circunscripción (desde `mapagan`)

In [17]:
# === Tabla 1: ca_candidatos_depto ===
# Votos de cada candidato por partido, departamento y circunscripción
filas_candidatos = []

for depto_codigo, resp in responses_ca.items():
    depto_nombre = dim_departamentos[
        dim_departamentos['depto_codigo'] == depto_codigo
    ]['depto_nombre'].values[0]
    
    for cam in resp['camaras']:
        cam_id = cam['cam']
        circunscripcion = CIRCUNSCRIPCIONES.get(cam_id, f'Desconocida ({cam_id})')
        
        for partido in cam.get('partotabla', []):
            p = partido['act']
            codpar = p['codpar']
            votos_partido = p['vot']
            
            for candidato in p.get('cantotabla', []):
                filas_candidatos.append({
                    'depto_codigo': depto_codigo,
                    'depto_nombre': depto_nombre,
                    'circunscripcion_id': cam_id,
                    'circunscripcion': circunscripcion,
                    'partido_codigo': codpar,
                    'votos_partido': int(votos_partido),
                    'candidato_codigo': candidato['codcan'],
                    'candidato_cedula': candidato['cedula'],
                    'candidato_nombre': f"{candidato['nomcan']} {candidato['apecan']}".strip(),
                    'candidato_votos': int(candidato['vot']),
                    'candidato_porcentaje': candidato['pvot'],
                    'preferido': candidato.get('pref', '0'),
                })

ca_candidatos_depto = pd.DataFrame(filas_candidatos)

# Enriquecer con nombre del partido desde dim_partidos
# Nota: el codpar del API corresponde a partido_indice del nomenclátor, NO a partido_codigo
_lookup_partido = dim_partidos[['partido_indice', 'partido_nombre']].copy()
_lookup_partido['partido_indice'] = _lookup_partido['partido_indice'].astype(str)
ca_candidatos_depto = ca_candidatos_depto.merge(
    _lookup_partido,
    left_on='partido_codigo',
    right_on='partido_indice',
    how='left'
).drop(columns=['partido_indice'])

print(f"ca_candidatos_depto: {len(ca_candidatos_depto)} filas")
print(f"  Departamentos: {ca_candidatos_depto['depto_nombre'].nunique()}")
print(f"  Circunscripciones: {ca_candidatos_depto['circunscripcion'].nunique()}")
print(f"  Candidatos únicos: {ca_candidatos_depto['candidato_nombre'].nunique()}")

print(f"  Partidos: {ca_candidatos_depto['partido_nombre'].nunique()}")

print("\nFilas por circunscripción:")
print(ca_candidatos_depto.groupby('circunscripcion').size())
ca_candidatos_depto.head(10)

ca_candidatos_depto: 8776 filas
  Departamentos: 34
  Circunscripciones: 3
  Candidatos únicos: 1774
  Partidos: 171

Filas por circunscripción:
circunscripcion
Afrodescendientes            5780
Indígenas                    1054
Territorial Departamental    1942
dtype: int64


,depto_codigo,depto_nombre,circunscripcion_id,circunscripcion,partido_codigo,votos_partido,candidato_codigo,candidato_cedula,candidato_nombre,candidato_votos,candidato_porcentaje,preferido,partido_nombre
0,0500,BOLIVAR,1,Territorial Departamental,3,422632,101,45546190,MARIA CAMILA SALAS SALAS,140621,"15,38%",1,PARTIDO CONSERVADOR COLOMBIANO
1,0500,BOLIVAR,1,Territorial Departamental,3,422632,102,45564617,JULIANA ARAY FRANCO,85756,"9,38%",1,PARTIDO CONSERVADOR COLOMBIANO
2,0500,BOLIVAR,1,Territorial Departamental,3,422632,106,1047394814,ANDRES GUILLERMO MONTES CELEDON,58371,"6,38%",1,PARTIDO CONSERVADOR COLOMBIANO
3,0500,BOLIVAR,1,Territorial Departamental,3,422632,103,73082377,ALONSO JOSE DEL RIO CABARCAS,53570,"5,86%",1,PARTIDO CONSERVADOR COLOMBIANO
4,0500,BOLIVAR,1,Territorial Departamental,3,422632,105,1047474159,MIGUEL ALFONSO MONTES CURI,39714,"4,34%",1,PARTIDO CONSERVADOR COLOMBIANO
5,0500,BOLIVAR,1,Territorial Departamental,3,422632,104,1047480293,MARIA PAULA CABALLERO PORTO,24165,"2,64%",1,PARTIDO CONSERVADOR COLOMBIANO
6,0500,BOLIVAR,1,Territorial Departamental,3,422632,0,,SOLO POR LA LISTA,20435,"2,23%",1,PARTIDO CONSERVADOR COLOMBIANO
7,0500,BOLIVAR,1,Territorial Departamental,81,170882,0,,SOLO POR LA LISTA,170882,"18,69%",0,PACTO HISTÓRICO BOLÍVAR
8,0500,BOLIVAR,1,Territorial Departamental,81,170882,101,73109223,AMAURY JULIO PEREZ,0,0%,0,PACTO HISTÓRICO BOLÍVAR
9,0500,BOLIVAR,1,Territorial Departamental,81,170882,102,64542016,COLOMBIA ESPERANZA ADUEN BRAY,0,0%,0,PACTO HISTÓRICO BOLÍVAR


In [30]:
# === Tabla 2: ca_municipios_ganador ===
# Partido ganador y votos por municipio y circunscripción (desde mapagan)
filas_municipios = []

for depto_codigo, resp in responses_ca.items():
    depto_nombre = dim_departamentos[
        dim_departamentos['depto_codigo'] == depto_codigo
    ]['depto_nombre'].values[0]
    
    for cam in resp['camaras']:
        cam_id = cam['cam']
        circunscripcion = CIRCUNSCRIPCIONES.get(cam_id, f'Desconocida ({cam_id})')
        
        for mpio in cam.get('mapagan', []):
            filas_municipios.append({
                'depto_codigo': depto_codigo,
                'depto_nombre': depto_nombre,
                'circunscripcion_id': cam_id,
                'circunscripcion': circunscripcion,
                'mpio_codigo': mpio['amb'],
                'mpio_nombre': mpio.get('nombre', ''),
                'ganador_partido_codigo': str(mpio['codpar']), # Aseguramos string desde el origen
                'ganador_votos': int(mpio['vot']),
                'ganador_porcentaje': mpio['pvot'],
                'total_votantes': int(mpio['votant']),
                'participacion': mpio['pvotant'],
                'votos_validos': int(mpio['votcan']),
                'mesas_escrutadas': int(mpio['mesesc']),
                'pct_mesas_escrutadas': mpio['pmesesc'],
                'hay_empate': mpio.get('hayEmpate', '0'),
            })

ca_municipios_ganador = pd.DataFrame(filas_municipios)

# --- AJUSTE DEL MERGE ---
# Preparamos la tabla de dimensiones asegurando que el índice sea string
dim_partidos_clean = dim_partidos[['partido_indice', 'partido_nombre']].copy()
dim_partidos_clean['partido_indice'] = dim_partidos_clean['partido_indice'].astype(str)

ca_municipios_ganador = ca_municipios_ganador.merge(
    dim_partidos_clean,
    left_on='ganador_partido_codigo',
    right_on='partido_indice',
    how='left'
).drop(columns=['partido_indice'])

# Renombramos para claridad
ca_municipios_ganador = ca_municipios_ganador.rename(columns={'partido_nombre': 'ganador_partido_nombre'})

# --- REPORTES ---
print(f"ca_municipios_ganador: {len(ca_municipios_ganador)} filas")
print(f"  Departamentos: {ca_municipios_ganador['depto_nombre'].nunique()}")
print(f"  Municipios: {ca_municipios_ganador['mpio_nombre'].nunique()}")
print(f"  Circunscripciones: {ca_municipios_ganador['circunscripcion'].nunique()}")

print("\nFilas por circunscripción:")
print(ca_municipios_ganador.groupby('circunscripcion').size())

print("\nGanador por circunscripción (Top 3 partidos):")
# Usamos un groupby más limpio para el reporte
top_ganadores = ca_municipios_ganador.groupby(['circunscripcion', 'ganador_partido_nombre']).size().unstack(fill_value=0)

for circ in ca_municipios_ganador['circunscripcion'].unique():
    print(f"\n  {circ}:")
    counts = ca_municipios_ganador[ca_municipios_ganador['circunscripcion'] == circ]['ganador_partido_nombre'].value_counts().head(3)
    print(counts.to_string(header=False))

ca_municipios_ganador.head(10)

ca_municipios_ganador: 3567 filas
  Departamentos: 34
  Municipios: 1100
  Circunscripciones: 3

Filas por circunscripción:
circunscripcion
Afrodescendientes            1189
Indígenas                    1189
Territorial Departamental    1189
dtype: int64

Ganador por circunscripción (Top 3 partidos):

  Territorial Departamental:
PARTIDO CONSERVADOR COLOMBIANO    223
PARTIDO LIBERAL COLOMBIANO        209
PARTIDO CENTRO DEMOCRÁTICO        173

  Indígenas:
MOVIMIENTO ALTERNATIVO INDÍGENA Y SOCIAL "MAIS"    886
MOVIMIENTO UNIDAD EN MINGA POR COLOMBIA            131
PARTIDO INDÍGENA COLOMBIANO P.I.C                   65

  Afrodescendientes:
CONSEJO COMUNITARIO EL NARANJO    655
PARTIDO ECOLOGISTA COLOMBIANO     168
PARTIDO DEMÓCRATA COLOMBIANO       85


,depto_codigo,depto_nombre,circunscripcion_id,circunscripcion,mpio_codigo,mpio_nombre,ganador_partido_codigo,ganador_votos,ganador_porcentaje,total_votantes,participacion,votos_validos,mesas_escrutadas,pct_mesas_escrutadas,hay_empate,ganador_partido_nombre
0,0500,BOLIVAR,1,Territorial Departamental,0500004,ACHI,3,6661,"57,95%",12030,"63,39%",11444,63,100%,0,PARTIDO CONSERVADOR COLOMBIANO
1,0500,BOLIVAR,1,Territorial Departamental,0500006,ALTOS DEL ROSARIO,3,3186,"68,26%",5039,"59,09%",4642,27,100%,0,PARTIDO CONSERVADOR COLOMBIANO
2,0500,BOLIVAR,1,Territorial Departamental,0500005,ARENAL,3,2435,"66,24%",3938,"55,78%",3612,25,100%,0,PARTIDO CONSERVADOR COLOMBIANO
3,0500,BOLIVAR,1,Territorial Departamental,0500007,ARJONA,3,13332,"43,26%",33204,"55,12%",29913,175,100%,0,PARTIDO CONSERVADOR COLOMBIANO
4,0500,BOLIVAR,1,Territorial Departamental,0500009,ARROYO HONDO,3,5861,"86,77%",7078,"67,23%",6732,33,100%,0,PARTIDO CONSERVADOR COLOMBIANO
5,0500,BOLIVAR,1,Territorial Departamental,0500010,BARRANCO DE LOBA,3,4526,"66,10%",7345,"55,90%",6784,41,100%,0,PARTIDO CONSERVADOR COLOMBIANO
6,0500,BOLIVAR,1,Territorial Departamental,0500013,CALAMAR,3,9047,"69,12%",14038,"66,41%",12983,64,100%,0,PARTIDO CONSERVADOR COLOMBIANO
7,0500,BOLIVAR,1,Territorial Departamental,0500014,CANTAGALLO,3,2913,"53,47%",5714,"68,53%",5402,26,100%,0,PARTIDO CONSERVADOR COLOMBIANO
8,0500,BOLIVAR,1,Territorial Departamental,0500001,CARTAGENA,3,118393,"30,82%",407889,"47,13%",370063,2638,"99,88%",0,PARTIDO CONSERVADOR COLOMBIANO
9,0500,BOLIVAR,1,Territorial Departamental,0500015,CICUCO,3,4775,"71,93%",6991,"59,59%",6603,36,100%,0,PARTIDO CONSERVADOR COLOMBIANO


In [31]:
# === Resumen: votos totales por candidato a nivel nacional (por circunscripción) ===
for circ in ['Territorial Departamental', 'Indígenas', 'Afrodescendientes']:
    subset = ca_candidatos_depto[ca_candidatos_depto['circunscripcion'] == circ]
    resumen = subset.groupby(['partido_nombre', 'candidato_nombre']).agg(
        votos_totales=('candidato_votos', 'sum'),
        deptos_presencia=('depto_codigo', 'nunique'),
    ).sort_values('votos_totales', ascending=False).reset_index()
    
    print(f"\n{'='*80}")
    print(f"Top 10 candidatos - {circ} ({len(resumen)} candidatos, {subset['candidato_votos'].sum():,} votos totales)")
    print(f"{'='*80}")
    for _, row in resumen.head(10).iterrows():
        print(f"  {row['votos_totales']:>10,}  {row['candidato_nombre'][:40]:<42s} ({row['partido_nombre'][:30]})")


Top 10 candidatos - Territorial Departamental (1746 candidatos, 18,283,906 votos totales)
   2,147,662  SOLO POR LA LISTA                          (MOVIMIENTO POLÍTICO PACTO HIST)
     867,154  SOLO POR LA LISTA                          (PARTIDO CENTRO DEMOCRÁTICO)
     394,081  SOLO POR LA LISTA                          (PACTO HISTÓRICO ANTIOQUIA)
     269,814  SOLO POR LA LISTA                          (PACTO HISTÓRICO ATLÁNTICO)
     262,104  DANIEL FELIPE BRICEÑO MONTES               (PARTIDO CENTRO DEMOCRÁTICO)
     223,879  SOLO POR LA LISTA                          (PARTIDO CAMBIO RADICAL)
     174,598  SOLO POR LA LISTA                          (PACTO HISTÓRICO CÓRDOBA)
     170,882  SOLO POR LA LISTA                          (PACTO HISTÓRICO BOLÍVAR)
     168,103  SOLO POR LA LISTA                          (PARTIDO LIBERAL COLOMBIANO)
     149,126  SOLO POR LA LISTA                          (PARTIDO DE LA UNIÓN POR LA GEN)

Top 10 candidatos - Indígenas (31 candidatos, 107,52

## Exportación de datos

Disponibilizamos los 2 DataFrames en múltiples formatos dentro de `data/raw_data_extract/camara/`:

| DataFrame | Filas | Descripción |
|-----------|-------|-------------|
| `ca_candidatos_depto` | 8,776 | Votos de cada candidato por partido, departamento y circunscripción |
| `ca_municipios_ganador` | 3,567 | Partido ganador por municipio y circunscripción |

| Formato | Archivo | Uso típico |
|---------|---------|------------|
| CSV (coma) | `.csv` | Compatible universal, Excel, R, Python |
| CSV (punto y coma) | `_semicolon.csv` | Excel en configuraciones regionales con coma decimal |
| Excel | `.xlsx` | Análisis rápido en hojas de cálculo |
| JSON | `.json` | APIs, JavaScript, visualizaciones web |
| Parquet | `.parquet` | Alto rendimiento para grandes volúmenes, Spark, pandas |

In [32]:
import os

# Directorio de salida (relativo a la raíz del proyecto)
OUTPUT_DIR = os.path.join('..', '..', 'data', 'raw_data_extract', 'camara')
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Exportar ambos DataFrames
datasets = {
    'ca_candidatos_depto': ca_candidatos_depto,
    'ca_municipios_ganador': ca_municipios_ganador,
}

for nombre, df in datasets.items():
    base = os.path.join(OUTPUT_DIR, nombre)
    
    # CSV (separado por coma)
    df.to_csv(f'{base}.csv', index=False, encoding='utf-8-sig')
    
    # CSV (separado por punto y coma)
    df.to_csv(f'{base}_semicolon.csv', index=False, encoding='utf-8-sig', sep=';')
    
    # Excel
    df.to_excel(f'{base}.xlsx', index=False, engine='openpyxl')
    
    # JSON
    df.to_json(f'{base}.json', orient='records', force_ascii=False, indent=2)
    
    # Parquet
    df.to_parquet(f'{base}.parquet', index=False, engine='pyarrow')
    
    print(f"\n✅ {nombre} ({len(df)} filas) — 5 archivos exportados:")
    for ext in ['.csv', '_semicolon.csv', '.xlsx', '.json', '.parquet']:
        filepath = f'{base}{ext}'
        size_mb = os.path.getsize(filepath) / (1024 * 1024)
        print(f"   {os.path.basename(filepath):45s} {size_mb:.2f} MB")


✅ ca_candidatos_depto (8776 filas) — 5 archivos exportados:
   ca_candidatos_depto.csv                       1.03 MB
   ca_candidatos_depto_semicolon.csv             1.03 MB
   ca_candidatos_depto.xlsx                      0.62 MB
   ca_candidatos_depto.json                      3.73 MB
   ca_candidatos_depto.parquet                   0.13 MB

✅ ca_municipios_ganador (3567 filas) — 5 archivos exportados:
   ca_municipios_ganador.csv                     0.44 MB
   ca_municipios_ganador_semicolon.csv           0.43 MB
   ca_municipios_ganador.xlsx                    0.28 MB
   ca_municipios_ganador.json                    1.79 MB
   ca_municipios_ganador.parquet                 0.10 MB


## Scraping a nivel municipal: votos por candidato en cada municipio

**Hallazgo clave**: Los endpoints de 7 dígitos (`/json/ACT/CA/{ambito_codigo}.json`) SÍ devuelven datos de candidatos a nivel municipal — con las 3 circunscripciones. Los códigos de 7 dígitos corresponden al `ambito_codigo` de `dim_ambitos` (nivel 3 = municipio).

**Estrategia**:
1. Consultar los **1,189 municipios** individualmente con `ThreadPoolExecutor` + `requests.Session`
2. De cada response extraer `partotabla` → candidatos por partido, por cada circunscripción
3. Construir `ca_candidatos_municipio` con la máxima granularidad

In [33]:
# === Scraping paralelo de los 1,189 municipios para Cámara (CA) ===
MAX_WORKERS_MPIO = 20

# Municipios: nivel_id == 3 en dim_ambitos (código de 7 dígitos)
municipios = dim_ambitos[dim_ambitos['nivel_id'] == 3].copy()
total = len(municipios)
print(f"Municipios a consultar: {total}")

# Session con connection pool
session_mpio = requests.Session()
session_mpio.cookies.update(cookies)
session_mpio.headers.update(headers)
adapter_mpio = HTTPAdapter(pool_connections=MAX_WORKERS_MPIO, pool_maxsize=MAX_WORKERS_MPIO)
session_mpio.mount('https://', adapter_mpio)

lock_mpio = threading.Lock()
completados_mpio = 0

def fetch_municipio_ca(row):
    """Descarga y parsea los candidatos de Cámara de un municipio."""
    global completados_mpio
    codigo = row['ambito_codigo']
    nombre = row['ambito_nombre']
    depto = row['departamento_nombre']
    
    r = session_mpio.get(f"{BASE_URL}/{codigo}.json")
    
    with lock_mpio:
        completados_mpio += 1
        if completados_mpio % 200 == 0:
            print(f"  [{completados_mpio:4d}/{total}] procesados...")
    
    if r.status_code != 200:
        return [], (codigo, nombre, r.status_code)
    
    resp = r.json()
    filas = []
    
    for cam in resp.get('camaras', []):
        cam_id = cam['cam']
        circunscripcion = CIRCUNSCRIPCIONES.get(cam_id, f'Desconocida ({cam_id})')
        totales = cam.get('totales', {}).get('act', {})
        
        for partido in cam.get('partotabla', []):
            p = partido['act']
            for candidato in p.get('cantotabla', []):
                filas.append({
                    'depto_nombre': depto,
                    'mpio_codigo': codigo,
                    'mpio_nombre': nombre,
                    'circunscripcion_id': cam_id,
                    'circunscripcion': circunscripcion,
                    'partido_codigo': p['codpar'],
                    'votos_partido_mpio': int(p['vot']),
                    'candidato_codigo': candidato['codcan'],
                    'candidato_cedula': candidato['cedula'],
                    'candidato_nombre': f"{candidato['nomcan']} {candidato['apecan']}".strip(),
                    'candidato_votos': int(candidato['vot']),
                    'candidato_porcentaje': candidato['pvot'],
                    'preferido': candidato.get('pref', '0'),
                    'mpio_votantes': int(totales.get('votant', 0)),
                    'mpio_votos_validos': int(totales.get('votval', 0)),
                    'mpio_votos_nulos': int(totales.get('votnul', 0)),
                    'mpio_votos_no_marcados': int(totales.get('votnma', 0)),
                })
    return filas, None

rows_mpio = [row for _, row in municipios.iterrows()]
filas_ca_mpio = []
errores_mpio = []

print(f"Descargando datos de {total} municipios ({MAX_WORKERS_MPIO} workers concurrentes)...")
t0 = time.time()
completados_mpio = 0

with ThreadPoolExecutor(max_workers=MAX_WORKERS_MPIO) as executor:
    futures = {executor.submit(fetch_municipio_ca, row): row for row in rows_mpio}
    for future in as_completed(futures):
        filas, error = future.result()
        filas_ca_mpio.extend(filas)
        if error:
            errores_mpio.append(error)

elapsed = time.time() - t0
print(f"\n✅ Completado en {elapsed:.1f}s: {total - len(errores_mpio)} OK, {len(errores_mpio)} errores")
print(f"   Filas extraídas: {len(filas_ca_mpio):,}")
if errores_mpio:
    print(f"   Errores: {errores_mpio[:5]}")


Municipios a consultar: 1189
Descargando datos de 1189 municipios (20 workers concurrentes)...
  [ 200/1189] procesados...
  [ 400/1189] procesados...
  [ 600/1189] procesados...
  [ 800/1189] procesados...
  [1000/1189] procesados...

✅ Completado en 18.8s: 1189 OK, 0 errores
   Filas extraídas: 327,104


In [34]:
# === DataFrame final: ca_candidatos_municipio ===
ca_candidatos_municipio = pd.DataFrame(filas_ca_mpio)

# 1. Aseguramos que la columna de unión sea string para evitar fallos de coincidencia
ca_candidatos_municipio['partido_codigo'] = ca_candidatos_municipio['partido_codigo'].astype(str)

# 2. Preparar el lookup de partidos (asegurando también el tipo string)
_lookup_partido = dim_partidos[['partido_indice', 'partido_nombre']].copy()
_lookup_partido['partido_indice'] = _lookup_partido['partido_indice'].astype(str)

# 3. Realizar el merge
ca_candidatos_municipio = ca_candidatos_municipio.merge(
    _lookup_partido,
    left_on='partido_codigo',
    right_on='partido_indice',
    how='left'
).drop(columns=['partido_indice'])

# --- Reportes y Verificación ---
print(f"ca_candidatos_municipio: {len(ca_candidatos_municipio):,} filas × {len(ca_candidatos_municipio.columns)} columnas")
print(f"  Departamentos: {ca_candidatos_municipio['depto_nombre'].nunique()}")
print(f"  Municipios: {ca_candidatos_municipio['mpio_nombre'].nunique()}")
print(f"  Circunscripciones: {ca_candidatos_municipio['circunscripcion'].nunique()}")
print(f"  Candidatos únicos: {ca_candidatos_municipio['candidato_nombre'].nunique()}")
print(f"  Partidos: {ca_candidatos_municipio['partido_nombre'].nunique()}")

# Verificación rápida de si quedaron partidos sin nombre (cruce fallido)
nulos = ca_candidatos_municipio['partido_nombre'].isna().sum()
if nulos > 0:
    print(f"\n⚠️ ATENCIÓN: Hay {nulos} filas sin nombre de partido. Revisa si hay códigos nuevos.")

print("\nFilas por circunscripción:")
print(ca_candidatos_municipio.groupby('circunscripcion').size())

print(f"\nColumnas finales: {list(ca_candidatos_municipio.columns)}")
ca_candidatos_municipio.head(10)

ca_candidatos_municipio: 327,104 filas × 18 columnas
  Departamentos: 34
  Municipios: 1100
  Circunscripciones: 3
  Candidatos únicos: 1774
  Partidos: 171

Filas por circunscripción:
circunscripcion
Afrodescendientes            202130
Indígenas                     36859
Territorial Departamental     88115
dtype: int64

Columnas finales: ['depto_nombre', 'mpio_codigo', 'mpio_nombre', 'circunscripcion_id', 'circunscripcion', 'partido_codigo', 'votos_partido_mpio', 'candidato_codigo', 'candidato_cedula', 'candidato_nombre', 'candidato_votos', 'candidato_porcentaje', 'preferido', 'mpio_votantes', 'mpio_votos_validos', 'mpio_votos_nulos', 'mpio_votos_no_marcados', 'partido_nombre']


,depto_nombre,mpio_codigo,mpio_nombre,circunscripcion_id,circunscripcion,partido_codigo,votos_partido_mpio,candidato_codigo,candidato_cedula,candidato_nombre,candidato_votos,candidato_porcentaje,preferido,mpio_votantes,mpio_votos_validos,mpio_votos_nulos,mpio_votos_no_marcados,partido_nombre
0,NARIÑO,2300004,ALBAN (SAN JOSE),1,Territorial Departamental,71,1168,101,36950716,ALEJANDRA GABRIELA ABASOLO GOMEZ,990,"21,44%",1,4925,4617,227,81,AVANCEMOS NARIÑO
1,NARIÑO,2300004,ALBAN (SAN JOSE),1,Territorial Departamental,71,1168,102,87215438,EDGAR FERNANDO RAMIREZ BRAVO,88,"1,90%",1,4925,4617,227,81,AVANCEMOS NARIÑO
2,NARIÑO,2300004,ALBAN (SAN JOSE),1,Territorial Departamental,71,1168,105,1088592421,CARLOS ANDRES CUAICAL CHAPI,34,"0,73%",1,4925,4617,227,81,AVANCEMOS NARIÑO
3,NARIÑO,2300004,ALBAN (SAN JOSE),1,Territorial Departamental,71,1168,0,,SOLO POR LA LISTA,33,"0,71%",1,4925,4617,227,81,AVANCEMOS NARIÑO
4,NARIÑO,2300004,ALBAN (SAN JOSE),1,Territorial Departamental,71,1168,103,36751142,DEY YAMA CORDOBA,22,"0,47%",1,4925,4617,227,81,AVANCEMOS NARIÑO
5,NARIÑO,2300004,ALBAN (SAN JOSE),1,Territorial Departamental,71,1168,104,13066016,ROBERT WILSON SALDAÑA BASANTE,1,"0,02%",1,4925,4617,227,81,AVANCEMOS NARIÑO
6,NARIÑO,2300004,ALBAN (SAN JOSE),1,Territorial Departamental,2,693,101,98390026,JULIO ANIBAL ALVAREZ LOPEZ,626,"13,55%",1,4925,4617,227,81,PARTIDO LIBERAL COLOMBIANO
7,NARIÑO,2300004,ALBAN (SAN JOSE),1,Territorial Departamental,2,693,0,,SOLO POR LA LISTA,32,"0,69%",1,4925,4617,227,81,PARTIDO LIBERAL COLOMBIANO
8,NARIÑO,2300004,ALBAN (SAN JOSE),1,Territorial Departamental,2,693,103,98322218,SERGIO ANTONIO MUÑOZ CASTILLO,15,"0,32%",1,4925,4617,227,81,PARTIDO LIBERAL COLOMBIANO
9,NARIÑO,2300004,ALBAN (SAN JOSE),1,Territorial Departamental,2,693,105,98381525,ALVARO JAVIER ZARAMA BURBANO,12,"0,25%",1,4925,4617,227,81,PARTIDO LIBERAL COLOMBIANO


In [37]:
# === Resumen: Top candidatos por circunscripción (agregado desde municipios) ===
for circ in ['Territorial Departamental', 'Indígenas', 'Afrodescendientes']:
    subset = ca_candidatos_municipio[ca_candidatos_municipio['circunscripcion'] == circ]
    resumen = subset.groupby(['partido_nombre', 'candidato_nombre']).agg(
        votos_totales=('candidato_votos', 'sum'),
        mpios_presencia=('mpio_codigo', 'nunique'),
    ).sort_values('votos_totales', ascending=False).reset_index()
    
    print(f"\n{'='*90}")
    print(f"Top 10 - {circ} ({len(resumen)} candidatos, {subset['candidato_votos'].sum():,} votos)")
    print(f"{'='*90}")
    for _, row in resumen.head(10).iterrows():
        print(f"  {row['votos_totales']:>10,}  {row['candidato_nombre'][:40]:<42s} ({row['partido_nombre'][:35]}) [{row['mpios_presencia']} mpios]")


Top 10 - Territorial Departamental (1746 candidatos, 18,283,906 votos)
   2,147,662  SOLO POR LA LISTA                          (MOVIMIENTO POLÍTICO PACTO HISTÓRICO) [332 mpios]
     867,154  SOLO POR LA LISTA                          (PARTIDO CENTRO DEMOCRÁTICO) [925 mpios]
     394,081  SOLO POR LA LISTA                          (PACTO HISTÓRICO ANTIOQUIA) [125 mpios]
     269,814  SOLO POR LA LISTA                          (PACTO HISTÓRICO ATLÁNTICO) [23 mpios]
     262,104  DANIEL FELIPE BRICEÑO MONTES               (PARTIDO CENTRO DEMOCRÁTICO) [1 mpios]
     223,879  SOLO POR LA LISTA                          (PARTIDO CAMBIO RADICAL) [460 mpios]
     174,598  SOLO POR LA LISTA                          (PACTO HISTÓRICO CÓRDOBA) [30 mpios]
     170,882  SOLO POR LA LISTA                          (PACTO HISTÓRICO BOLÍVAR) [46 mpios]
     168,103  SOLO POR LA LISTA                          (PARTIDO LIBERAL COLOMBIANO) [1048 mpios]
     149,126  SOLO POR LA LISTA                      

## Verificación de consistencia: municipal vs departamental

Validamos que los votos totales agregados desde municipios coincidan con los datos departamentales. Esto asegura que el scraping municipal es correcto y completo.

In [38]:
# === Verificación: ¿los votos municipales suman igual que los departamentales? ===
print("Verificación de consistencia: municipal vs departamental")
print("=" * 70)

for circ in ['Territorial Departamental', 'Indígenas', 'Afrodescendientes']:
    # Total desde datos departamentales
    subset_depto = ca_candidatos_depto[ca_candidatos_depto['circunscripcion'] == circ]
    total_depto = subset_depto['candidato_votos'].sum()
    
    # Total desde datos municipales
    subset_mpio = ca_candidatos_municipio[ca_candidatos_municipio['circunscripcion'] == circ]
    total_mpio = subset_mpio['candidato_votos'].sum()
    
    match = "✅" if total_depto == total_mpio else "⚠️ DIFERENCIA"
    print(f"  {circ}:")
    print(f"    Departamental: {total_depto:>12,} votos")
    print(f"    Municipal:     {total_mpio:>12,} votos")
    print(f"    {match}")
    print()

# Verificar cobertura de municipios
mpios_esperados = dim_ambitos[dim_ambitos['nivel_id'] == 3]['ambito_codigo'].nunique()
mpios_obtenidos = ca_candidatos_municipio['mpio_codigo'].nunique()
print(f"Municipios esperados: {mpios_esperados}")
print(f"Municipios obtenidos: {mpios_obtenidos}")
print(f"Cobertura: {mpios_obtenidos/mpios_esperados*100:.1f}%")


Verificación de consistencia: municipal vs departamental
  Territorial Departamental:
    Departamental:   18,283,906 votos
    Municipal:       18,283,906 votos
    ✅

  Indígenas:
    Departamental:      107,520 votos
    Municipal:          107,520 votos
    ✅

  Afrodescendientes:
    Departamental:      450,229 votos
    Municipal:          450,229 votos
    ✅

Municipios esperados: 1189
Municipios obtenidos: 1189
Cobertura: 100.0%


## Exportación de datos (máxima granularidad municipal)

Exportamos `ca_candidatos_municipio` — votos de cada candidato de cada partido en cada municipio, por cada circunscripción — en 5 formatos a `data/raw_data_extract/camara/`.

| Formato | Archivo | Uso típico |
|---------|---------|------------|
| CSV (coma) | `.csv` | Compatible universal |
| CSV (punto y coma) | `_semicolon.csv` | Excel con coma decimal |
| Excel | `.xlsx` | Hojas de cálculo |
| JSON | `.json` | APIs, visualizaciones web |
| Parquet | `.parquet` | pandas, PySpark, DuckDB |

In [39]:
import os

OUTPUT_DIR = os.path.join('..', '..', 'data', 'raw_data_extract', 'camara')
os.makedirs(OUTPUT_DIR, exist_ok=True)

base = os.path.join(OUTPUT_DIR, 'ca_candidatos_municipio')

# CSV (separado por coma)
ca_candidatos_municipio.to_csv(f'{base}.csv', index=False, encoding='utf-8-sig')

# CSV (separado por punto y coma)
ca_candidatos_municipio.to_csv(f'{base}_semicolon.csv', index=False, encoding='utf-8-sig', sep=';')

# Excel (skip if > 1,048,576 rows — Excel limit)
if len(ca_candidatos_municipio) <= 1_048_576:
    ca_candidatos_municipio.to_excel(f'{base}.xlsx', index=False, engine='openpyxl')
else:
    print(f"⚠️  XLSX omitido: {len(ca_candidatos_municipio):,} filas excede el límite de Excel (1,048,576)")

# JSON
ca_candidatos_municipio.to_json(f'{base}.json', orient='records', force_ascii=False, indent=2)

# Parquet
ca_candidatos_municipio.to_parquet(f'{base}.parquet', index=False, engine='pyarrow')

formatos = ['.csv', '_semicolon.csv', '.json', '.parquet']
if len(ca_candidatos_municipio) <= 1_048_576:
    formatos.insert(2, '.xlsx')
print(f"\n✅ ca_candidatos_municipio ({len(ca_candidatos_municipio):,} filas) — {len(formatos)} archivos exportados:")
for ext in formatos:
    filepath = f'{base}{ext}'
    size_mb = os.path.getsize(filepath) / (1024 * 1024)
    print(f"   {os.path.basename(filepath):45s} {size_mb:.2f} MB")



✅ ca_candidatos_municipio (327,104 filas) — 5 archivos exportados:
   ca_candidatos_municipio.csv                   45.72 MB
   ca_candidatos_municipio_semicolon.csv         45.54 MB
   ca_candidatos_municipio.xlsx                  27.74 MB
   ca_candidatos_municipio.json                  185.83 MB
   ca_candidatos_municipio.parquet               1.46 MB


## Nota sobre el join de partidos

> **⚠️ Verificar manualmente**: En este notebook el `codpar` del API se matchea con `partido_indice` del nomenclátor (campo `i` de `dim_partidos`), NO con `partido_codigo` (campo `codpar`). 
>
> Si los nombres de partido aparecen como NaN o incorrectos, invertir el join: usar `partido_codigo` en vez de `partido_indice`.
>
> El notebook de Consultas usa `partido_codigo` para el mismo join. Uno de los dos enfoques puede estar equivocado dependiendo de cómo la Registraduría asigna los códigos para cada elección. Verificar con un ejemplo conocido.